In [1]:
# ==========================================================
# CELL 1: SETUP & ADVANCED FEATURE FUNCTIONS
# ==========================================================

print("--- Step 1 of 5: Setting up the environment ---")
!pip install -q transformers accelerate sentencepiece xgboost
import pandas as pd, numpy as np, re, os, gc, tensorflow as tf, joblib, lightgbm as lgb, xgboost as xgb
from tqdm.notebook import tqdm
from transformers import AutoTokenizer, TFAutoModelForSequenceClassification
from sklearn.model_selection import train_test_split

# --- Mount Google Drive & Define Paths ---
from google.colab import drive
drive.mount('/content/drive')
DRIVE_PATH = '/content/drive/My Drive/'
COMPETITION_DATA_PATH = os.path.join(DRIVE_PATH, 'student_resource/dataset/')
# We will use the local Colab disk for images this time to speed up training
TRAIN_IMAGE_DIR = "/content/train_images/"
TEST_IMAGE_DIR = "/content/test_images/"

# --- HARDWARE VERIFICATION ---
!nvidia-smi

# --- ADVANCED FEATURE ENGINEERING FUNCTION ---
# This function will extract IPQ, and also any weights/volumes it can find.
def extract_numerical_features(text_series):
    features = []
    for text in tqdm(text_series.fillna(""), desc="Extracting Numerical Feats"):
        text = text.lower()
        # Pack Size
        ipq_match = re.search(r'pack of (\d+)|(\d+)\s*count', text)
        ipq = int(ipq_match.group(1) or ipq_match.group(2)) if ipq_match else 1
        # Weight/Volume (oz, lb, g, kg, ml, l, fl oz)
        weight_match = re.search(r'(\d+\.?\d*)\s*(oz|ounce|lb|pound|g|gram|kg|ml|l|fl oz)\b', text)
        weight = float(weight_match.group(1)) if weight_match else 0
        features.append([ipq, weight])
    return np.array(features)

print("\n" + "="*50); print("  Step 1 COMPLETE. Ready for the main event.  "); print("="*50)

--- Step 1 of 5: Setting up the environment ---
Mounted at /content/drive
Mon Oct 13 11:43:21 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   35C    P0             55W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                       

In [3]:
# ==========================================================
# CELL 2: FINE-TUNE THE SOTA VISION EXPERT (EFFICIENTNETB3)
# ==========================================================



# --- Imports and Helper Functions for this cell ---
from concurrent.futures import ThreadPoolExecutor
import requests # Import the requests library

def download_image(args):
    url, sample_id, folder = args
    if not os.path.exists(folder): os.makedirs(folder)
    image_path = os.path.join(folder, f"{sample_id}.jpg")
    if not os.path.exists(image_path):
        try:
            r = requests.get(url, timeout=20); r.raise_for_status()
            with open(image_path, 'wb') as f: f.write(r.content)
        except: pass

# --- A100 OPTIMIZATION: ENABLE MIXED PRECISION ---
# This can dramatically speed up training on A100/V100 GPUs
print("\nEnabling Mixed Precision for A100 GPU acceleration...")
tf.keras.mixed_precision.set_global_policy('mixed_float16')

# --- Load Data, Download Images, and Create TF.Data Pipeline ---
print("\n[2a] Loading data, downloading images, and preparing the data pipeline...")
train_df = pd.read_csv(os.path.join(COMPETITION_DATA_PATH, 'train.csv'))

# Download images to the fast local disk of the Colab machine
download_args = [(row.image_link, row.sample_id, TRAIN_IMAGE_DIR) for _, row in train_df.iterrows()]
with ThreadPoolExecutor(max_workers=16) as executor:
    list(tqdm(executor.map(download_image, download_args), total=len(download_args), desc="Downloading Train Images"))

train_df['image_path'] = train_df['sample_id'].apply(lambda x: os.path.join(TRAIN_IMAGE_DIR, f"{x}.jpg"))
train_df['image_exists'] = train_df['image_path'].apply(os.path.exists)
train_df = train_df[train_df['image_exists']].copy()
y_train_log = np.log1p(train_df['price'])

X_train_img, X_val_img, y_train_img, y_val_img = train_test_split(
    train_df['image_path'].values, y_train_log.values, test_size=0.1, random_state=42
)

IMG_SIZE = 300 # EfficientNetB3 uses a larger image size
BATCH_SIZE = 64 # Larger batch size for the A100

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1)
])

def parse_image_aug(filepath, label):
    img = tf.io.read_file(filepath); img = tf.io.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE]); return img / 255.0, label

def configure_for_performance(ds, shuffle=True):
    # Caching the data in memory after the first epoch to speed up subsequent epochs
    ds = ds.map(parse_image_aug, num_parallel_calls=tf.data.AUTOTUNE).cache()
    if shuffle:
        ds = ds.shuffle(buffer_size=1000)
    ds = ds.batch(BATCH_SIZE)
    if shuffle:
        ds = ds.map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.prefetch(buffer_size=tf.data.AUTOTUNE)
    return ds

train_ds = configure_for_performance(tf.data.Dataset.from_tensor_slices((X_train_img, y_train_img)))
val_ds = configure_for_performance(tf.data.Dataset.from_tensor_slices((X_val_img, y_val_img)), shuffle=False)

# --- Build and Fine-Tune the Model ---
print("\n[2b] Building and Fine-Tuning the EfficientNetB3 model...")
strategy = tf.distribute.get_strategy()
with strategy.scope():
    # Load the more powerful B3 model
    base_model = tf.keras.applications.EfficientNetB3(include_top=False, weights='imagenet', input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base_model.trainable = True # Unfreeze the whole model for fine-tuning
    # Freeze the first blocks, fine-tune only the deeper, more specialized ones
    for layer in base_model.layers[:-50]:
        layer.trainable = False

    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = base_model(inputs, training=True)
    x = tf.keras.layers.GlobalAveragePooling2D(name="embedding_layer")(x)
    x = tf.keras.layers.Dropout(0.4)(x)
    # The final layer must be float32 for numerical stability, even in mixed precision
    outputs = tf.keras.layers.Dense(1, dtype='float32')(x)
    vision_model = tf.keras.Model(inputs, outputs)

    optimizer = tf.keras.optimizers.AdamW(learning_rate=1e-4) # Low learning rate for fine-tuning
    vision_model.compile(optimizer=optimizer, loss='mean_absolute_error')

vision_model.summary()

# --- Start Training ---
print("\n[2c] Starting the main training loop...")
callbacks = [tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True, verbose=1)]
vision_model.fit(train_ds, epochs=10, validation_data=val_ds, callbacks=callbacks)

# --- Save the SOTA Vision Model ---
vision_model.save(os.path.join(DRIVE_PATH, "sota_vision_model.keras"))
print("✅ Vision Expert model has been saved to your Google Drive.")

# --- CRITICAL MEMORY CLEANUP ---
del vision_model, base_model, train_ds, val_ds, train_df; gc.collect(); tf.keras.backend.clear_session()
print("\n" + "="*50); print("✅ Step 2 COMPLETE. SOTA Vision Expert is trained and saved."); print("="*50)


Enabling Mixed Precision for A100 GPU acceleration...

[2a] Loading data, downloading images, and preparing the data pipeline...



[2b] Building and Fine-Tuning the EfficientNetB3 model...
43941136/43941136 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 300, 300, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb3 (Functional)     │ (None, 10, 10, 1536)   │    10,783,535 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_layer                 │ (None, 1536)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1536)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │         1,537 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,785,072 (41.14 MB)

 Trainable params: 5,051,701 (19.27 MB)

 Non-trainable params: 5,733,371 (21.87 MB)


[2c] Starting the main training loop...
Epoch 1/10
1055/1055 ━━━━━━━━━━━━━━━━━━━━ 770s 585ms/step - loss: 0.8512 - val_loss: 0.7539
Epoch 2/10
1055/1055 ━━━━━━━━━━━━━━━━━━━━ 208s 195ms/step - loss: 0.7663 - val_loss: 0.7583
Epoch 3/10
1055/1055 ━━━━━━━━━━━━━━━━━━━━ 208s 195ms/step - loss: 0.7547 - val_loss: 0.7412
Epoch 4/10
1055/1055 ━━━━━━━━━━━━━━━━━━━━ 208s 195ms/step - loss: 0.7436 - val_loss: 0.7370
Epoch 5/10
1055/1055 ━━━━━━━━━━━━━━━━━━━━ 208s 195ms/step - loss: 0.7362 - val_loss: 0.7375
Epoch 6/10
1055/1055 ━━━━━━━━━━━━━━━━━━━━ 208s 195ms/step - loss: 0.7314 - val_loss: 0.7355
Epoch 7/10
1055/1055 ━━━━━━━━━━━━━━━━━━━━ 208s 196ms/step - loss: 0.7279 - val_loss: 0.7220
Epoch 8/10
1055/1055 ━━━━━━━━━━━━━━━━━━━━ 208s 195ms/step - loss: 0.7273 - val_loss: 0.7200
Epoch 9/10
1055/1055 ━━━━━━━━━━━━━━━━━━━━ 208s 195ms/step - loss: 0.7232 - val_loss: 0.7260
Epoch 10/10
1055/1055 ━━━━━━━━━━━━━━━━━━━━ 208s 195ms/step - loss: 0.7223 - val_loss: 0.7449
Epoch 10: early stopping
Restoring mod

In [10]:
# =====================================================================
# CELL 3  FINE-TUNE DEBERTA WITH CUSTOM TRAINING LOOP
# =====================================================================


# --- Self-Contained Setup for this Cell ---
import pandas as pd
import numpy as np
import os
import gc
import tensorflow as tf
from tqdm.notebook import tqdm
from transformers import AutoTokenizer, TFAutoModelForSequenceClassification
from sklearn.model_selection import train_test_split

# Re-define paths to be safe in a new session
DRIVE_PATH = '/content/drive/My Drive/'
COMPETITION_DATA_PATH = os.path.join(DRIVE_PATH, 'student_resource/dataset/')

# --- Load Data and Split ---
print("\n[3a] Loading data and preparing for Transformer...")
train_df = pd.read_csv(os.path.join(COMPETITION_DATA_PATH, 'train.csv'))
y_train_log = np.log1p(train_df['price'].values)
X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
    train_df['catalog_content'].fillna("").tolist(), y_train_log, test_size=0.1, random_state=42
)

# --- Tokenize the Text ---
print("\n[3b] Tokenizing text for the DeBERTa Transformer...")
MODEL_NAME = 'microsoft/deberta-v3-base'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
train_encodings = tokenizer(X_train_split, truncation=True, padding=True, max_length=256)
val_encodings = tokenizer(X_val_split, truncation=True, padding=True, max_length=256)

# --- Create TF Datasets ---
BATCH_SIZE = 16
train_dataset = tf.data.Dataset.from_tensor_slices((dict(train_encodings), y_train_split)).shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_dataset = tf.data.Dataset.from_tensor_slices((dict(val_encodings), y_val_split)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# --- Fine-Tune the Model with a Custom Training Loop ---
print("\n[3c] Setting up the model and custom training loop...")
strategy = tf.distribute.get_strategy()
with strategy.scope():
    language_model = TFAutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=1)
    optimizer = tf.keras.optimizers.AdamW(learning_rate=2e-5)
    loss_fn = tf.keras.losses.MeanSquaredError()

# --- THE UNBREAKABLE FIX: Custom Training Step ---
@tf.function
def train_step(dist_inputs):
    def step_fn(inputs):
        x, y = inputs
        with tf.GradientTape() as tape:
            outputs = language_model(x, training=True)
            # The loss is computed from the model's logits output
            loss = loss_fn(y, outputs.logits)

        grads = tape.gradient(loss, language_model.trainable_variables)
        optimizer.apply_gradients(zip(grads, language_model.trainable_variables))
        return loss
    per_replica_losses = strategy.run(step_fn, args=(dist_inputs,))
    return strategy.reduce(tf.distribute.ReduceOp.SUM, per_replica_losses, axis=None)

# --- Custom Validation Step ---
@tf.function
def val_step(dist_inputs):
    def step_fn(inputs):
        x, y = inputs
        outputs = language_model(x, training=False)
        loss = loss_fn(y, outputs.logits)
        return loss
    per_replica_losses = strategy.run(step_fn, args=(dist_inputs,))
    return strategy.reduce(tf.distribute.ReduceOp.SUM, per_replica_losses, axis=None)

# --- The Training Loop ---
print("\n[3d] Starting the main language model training loop...")
NUM_EPOCHS = 2
best_val_loss = float('inf')
patience = 1
patience_counter = 0

for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch {epoch + 1}/{NUM_EPOCHS}")
    total_loss = 0.0
    num_batches = 0

    # Training loop
    for batch in tqdm(train_dataset, desc="Training"):
        total_loss += train_step(batch)
        num_batches += 1
    train_loss = total_loss / num_batches
    print(f"Training Loss: {train_loss:.4f}")

    # Validation loop
    total_val_loss = 0.0
    num_val_batches = 0
    for batch in tqdm(val_dataset, desc="Validating"):
        total_val_loss += val_step(batch)
        num_val_batches += 1
    val_loss = total_val_loss / num_val_batches
    print(f"Validation Loss: {val_loss:.4f}")

    # Early stopping logic
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        print("Validation loss improved. Saving model...")
        language_model.save_pretrained(os.path.join(DRIVE_PATH, "sota_language_model"))
        tokenizer.save_pretrained(os.path.join(DRIVE_PATH, "sota_language_model"))
    else:
        patience_counter += 1
        print(f"Validation loss did not improve. Patience: {patience_counter}/{patience}")

    if patience_counter >= patience:
        print("Early stopping triggered.")
        break

# --- CRITICAL MEMORY CLEANUP ---
del language_model, train_dataset, val_dataset, train_df, X_train_split, X_val_split, y_train_split, y_val_split
gc.collect(); tf.keras.backend.clear_session()
print("\n" + "="*50); print("✅ Step 3 COMPLETE. SOTA Language Expert is trained and saved."); print("="*50)


[3a] Loading data and preparing for Transformer...

[3b] Tokenizing text for the DeBERTa Transformer...


/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(



[3c] Setting up the model and custom training loop...


All model checkpoint layers were used when initializing TFDebertaV2ForSequenceClassification.

Some layers of TFDebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier', 'pooler', 'cls_dropout']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



[3d] Starting the main language model training loop...

Epoch 1/2


...

Validation Loss: 0.5245
Validation loss improved. Saving model...

Epoch 2/2


Training:   0%|          | 0/4219 [00:00<?, ?it/s]

Training Loss: 0.4084


Validating:   0%|          | 0/469 [00:00<?, ?it/s]

Validation Loss: 0.4253
Validation loss improved. Saving model...

✅ Step 3 COMPLETE. SOTA Language Expert is trained and saved.


In [12]:
# ==========================================================
# CELL 4 (FINAL CORRECTED VERSION): GENERATE ALL SOTA FEATURES
# ==========================================================


# --- Load SOTA Models and Data ---
print("\n[4a] Loading all SOTA models and data...")
strategy = tf.distribute.get_strategy()
with strategy.scope():
    # Load the Vision Expert you trained in Cell 2
    vision_model = tf.keras.models.load_model(os.path.join(DRIVE_PATH, "sota_vision_model.keras"))
    vision_feature_extractor = tf.keras.Model(inputs=vision_model.input, outputs=vision_model.get_layer("embedding_layer").output)

    # Load the Language Expert you trained in Cell 3
    language_model = TFAutoModelForSequenceClassification.from_pretrained(os.path.join(DRIVE_PATH, "sota_language_model"))

    # --- THIS IS THE ONE-LINE FIX ---
    # The base model is the first layer itself, not an attribute of it.
    language_feature_extractor = language_model.layers[0]

    tokenizer = AutoTokenizer.from_pretrained(os.path.join(DRIVE_PATH, "sota_language_model"))

train_df = pd.read_csv(os.path.join(COMPETITION_DATA_PATH, 'train.csv'))
test_df = pd.read_csv(os.path.join(COMPETITION_DATA_PATH, 'test.csv'))
print("✅ All models and data loaded successfully.")

# --- Define Helper Functions for Feature Generation ---
def get_text_embeddings(texts, batch_size=64):
    all_embeddings = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Generating Text Embeddings"):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors='tf', truncation=True, padding=True, max_length=256)
        outputs = language_feature_extractor(inputs).last_hidden_state[:, 0, :]
        all_embeddings.append(outputs.numpy())
    return np.vstack(all_embeddings)

def parse_image_pred(filepath):
    try:
        img = tf.io.read_file(filepath); img = tf.io.decode_jpeg(img, channels=3)
        return tf.image.resize(img, [300, 300]) / 255.0 # Using 300x300 for B3 model
    except:
        return tf.zeros((300, 300, 3), dtype=tf.float32)

def get_image_embeddings(filepaths, batch_size=128):
    ds = tf.data.Dataset.from_tensor_slices(filepaths)
    ds = ds.map(parse_image_pred, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return vision_feature_extractor.predict(ds, verbose=1)

# --- Generate and Save All SOTA Features ---
print("\n[4b] Generating and saving all SOTA features to your Drive...")

# Advanced Numerical Features
np.save(os.path.join(DRIVE_PATH, 'sota_train_numerical_feats.npy'), extract_numerical_features(train_df['catalog_content']))
np.save(os.path.join(DRIVE_PATH, 'sota_test_numerical_feats.npy'), extract_numerical_features(test_df['catalog_content']))
print("✅ Numerical features saved.")

# SOTA Text Embeddings
np.save(os.path.join(DRIVE_PATH, 'sota_train_text_embeddings.npy'), get_text_embeddings(train_df['catalog_content'].fillna("").tolist()))
np.save(os.path.join(DRIVE_PATH, 'sota_test_text_embeddings.npy'), get_text_embeddings(test_df['catalog_content'].fillna("").tolist()))
print("✅ SOTA Text embeddings saved.")

# SOTA Image Embeddings
# Re-download test images to local storage for speed
from concurrent.futures import ThreadPoolExecutor
download_args_test = [(row.image_link, row.sample_id, TEST_IMAGE_DIR) for _, row in test_df.iterrows()]
with ThreadPoolExecutor(max_workers=16) as executor: list(tqdm(executor.map(download_image, download_args_test), total=len(download_args_test), desc="Downloading Test Images"))

train_df['image_path'] = train_df['sample_id'].apply(lambda x: os.path.join(TRAIN_IMAGE_DIR, f"{x}.jpg"))
test_df['image_path'] = test_df['sample_id'].apply(lambda x: os.path.join(TEST_IMAGE_DIR, f"{x}.jpg"))

np.save(os.path.join(DRIVE_PATH, 'sota_train_image_embeddings.npy'), get_image_embeddings(train_df[train_df['image_path'].apply(os.path.exists)]['image_path'].values))
np.save(os.path.join(DRIVE_PATH, 'sota_test_image_embeddings.npy'), get_image_embeddings(test_df[test_df['image_path'].apply(os.path.exists)]['image_path'].values))
print("✅ SOTA Image embeddings saved.")

# Save aligned IDs and labels for the final assembly
train_df_cleaned = train_df[train_df['image_path'].apply(os.path.exists)]
train_df_cleaned[['sample_id']].to_csv(os.path.join(DRIVE_PATH, 'sota_train_ids_aligned.csv'), index=False)
np.log1p(train_df_cleaned['price']).to_csv(os.path.join(DRIVE_PATH, 'sota_train_labels_log.csv'), index=False, header=True)
test_df[['sample_id']].to_csv(os.path.join(DRIVE_PATH, 'sota_test_ids_aligned.csv'), index=False)
print("✅ Final IDs and labels saved.")

# --- CRITICAL MEMORY CLEANUP ---
del vision_model, language_model, tokenizer; gc.collect(); tf.keras.backend.clear_session()
print("\n" + "="*50); print("✅ Step 4 COMPLETE. ALL SOTA features are permanently saved."); print("="*50)


[4a] Loading all SOTA models and data...


All model checkpoint layers were used when initializing TFDebertaV2ForSequenceClassification.

All the layers of TFDebertaV2ForSequenceClassification were initialized from the model checkpoint at /content/drive/My Drive/sota_language_model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFDebertaV2ForSequenceClassification for predictions without further training.


✅ All models and data loaded successfully.

[4b] Generating and saving all SOTA features to your Drive...


Extracting Numerical Feats:   0%|          | 0/75000 [00:00<?, ?it/s]

Extracting Numerical Feats:   0%|          | 0/75000 [00:00<?, ?it/s]

...

Generating Text Embeddings:   0%|          | 0/1172 [00:00<?, ?it/s]

TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.


Generating Text Embeddings:   0%|          | 0/1172 [00:00<?, ?it/s]

✅ SOTA Text embeddings saved.


586/586 ━━━━━━━━━━━━━━━━━━━━ 414s 541ms/step
586/586 ━━━━━━━━━━━━━━━━━━━━ 222s 378ms/step
✅ SOTA Image embeddings saved.
✅ Final IDs and labels saved.

✅ Step 4 COMPLETE. ALL SOTA features are permanently saved.


In [18]:
# ==========================================================
# CELL 5 (FINAL): ASSEMBLE, VALIDATE, AND SUBMIT
# ==========================================================


# --- Imports for this final cell ---
import pandas as pd
import numpy as np
import os
import gc
import lightgbm as lgb
import xgboost as xgb
from sklearn.model_selection import train_test_split

# --- Load All SOTA Features ---
print("\n[5a] Loading all of your SOTA feature files from Drive...")
train_text_emb = np.load(os.path.join(DRIVE_PATH, 'sota_train_text_embeddings.npy'))
test_text_emb = np.load(os.path.join(DRIVE_PATH, 'sota_test_text_embeddings.npy'))
train_image_emb = np.load(os.path.join(DRIVE_PATH, 'sota_train_image_embeddings.npy'))
test_image_emb = np.load(os.path.join(DRIVE_PATH, 'sota_test_image_embeddings.npy'))
train_num_feats = np.load(os.path.join(DRIVE_PATH, 'sota_train_numerical_feats.npy'))
test_num_feats = np.load(os.path.join(DRIVE_PATH, 'sota_test_numerical_feats.npy'))
y_train_log = pd.read_csv(os.path.join(DRIVE_PATH, 'sota_train_labels_log.csv')).iloc[:, 0].values
train_ids = pd.read_csv(os.path.join(DRIVE_PATH, 'sota_train_ids_aligned.csv'))
test_ids = pd.read_csv(os.path.join(DRIVE_PATH, 'sota_test_ids_aligned.csv'))

# --- CRITICAL ALIGNMENT STEP (TRAIN DATA) ---
print("\n[5b] Aligning all training features...")
original_train_df = pd.read_csv(os.path.join(COMPETITION_DATA_PATH, 'train.csv'))
alignment_mask_train = original_train_df['sample_id'].isin(train_ids['sample_id']).values
train_text_emb_aligned = train_text_emb[alignment_mask_train]
train_num_feats_aligned = train_num_feats[alignment_mask_train]
X_train_full = np.hstack([train_text_emb_aligned, train_image_emb, train_num_feats_aligned])
print(f"Alignment complete. Training features shape: {X_train_full.shape}")

# --- CRITICAL ALIGNMENT STEP (TEST DATA) ---
print("\n[5c] Aligning all test features...")
original_test_df = pd.read_csv(os.path.join(COMPETITION_DATA_PATH, 'test.csv'))
original_test_df['image_path'] = original_test_df['sample_id'].apply(lambda x: os.path.join(TEST_IMAGE_DIR, f"{x}.jpg"))
image_exists_mask_test = original_test_df['image_path'].apply(os.path.exists).values
aligned_test_image_emb = np.zeros((len(original_test_df), test_image_emb.shape[1]))
aligned_test_image_emb[image_exists_mask_test] = test_image_emb
X_test_full = np.hstack([test_text_emb, aligned_test_image_emb, test_num_feats])
print(f"Final test matrix shape: {X_test_full.shape}")

# --- Find Optimal Blend and Get SMAPE Score ---
print("\n[5d] Training ensemble and calculating your SOTA SMAPE score...")
X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(X_train_full, y_train_log, test_size=0.15, random_state=42)
def smape(y_true, y_pred): return np.mean(np.abs(y_pred - y_true) / ((np.abs(y_true) + np.abs(y_pred)) / 2)) * 100

lgbm_val = lgb.LGBMRegressor(random_state=42, n_estimators=4000, learning_rate=0.01, num_leaves=100, objective='regression_l1', n_jobs=-1, colsample_bytree=0.6, subsample=0.6)
lgbm_val.fit(X_train_split, y_train_split, eval_set=[(X_val_split, y_val_split)], callbacks=[lgb.early_stopping(200, verbose=False)])
lgbm_preds_val = lgbm_val.predict(X_val_split)

# --- THE UNBREAKABLE FIX FOR XGBOOST ---
# We remove early stopping from the validation model to guarantee it runs.
# The number of estimators is set to a powerful but safe value.
xgb_val = xgb.XGBRegressor(random_state=42, n_estimators=1200, learning_rate=0.015, max_depth=9, tree_method='gpu_hist', subsample=0.6, colsample_bytree=0.6, objective='reg:squarederror', n_jobs=-1)
xgb_val.fit(X_train_split, y_train_split, verbose=False) # No early stopping, just train for the full duration
xgb_preds_val = xgb_val.predict(X_val_split)

y_val_actual = np.expm1(y_val_split); best_smape = 100; best_weight = 0.5
for w in np.arange(0.0, 1.01, 0.01):
    blend = np.expm1(w * lgbm_preds_val + (1 - w) * xgb_preds_val)
    if smape(y_val_actual, blend) < best_smape: best_smape = smape(y_val_actual, blend); best_weight = w
print("\n" + "#"*50); print(f"##  FINAL SOTA ENSEMBLE SMAPE: {best_smape:.4f}%  ##"); print("#"*50)

# --- Train Final Models on 100% of Data ---
print("\n[5e] Training final models on 100% of data for submission...")
# We use the best iteration from the reliable LightGBM model for both
best_iteration_lgbm = lgbm_val.best_iteration_ if lgbm_val.best_iteration_ else 3000

final_lgbm = lgb.LGBMRegressor(random_state=42, n_estimators=best_iteration_lgbm, learning_rate=0.01, num_leaves=100, objective='regression_l1', n_jobs=-1, colsample_bytree=0.6, subsample=0.6)
final_lgbm.fit(X_train_full, y_train_log)

# We train XGBoost for a fixed, powerful number of rounds
final_xgb = xgb.XGBRegressor(random_state=42, n_estimators=1200, learning_rate=0.015, max_depth=9, tree_method='gpu_hist', subsample=0.6, colsample_bytree=0.6, objective='reg:squarederror', n_jobs=-1)
final_xgb.fit(X_train_full, y_train_log)

# --- Create Final Submission File ---
print("\n[5f] Blending predictions and creating your winning submission file...")
lgbm_preds_test = final_lgbm.predict(X_test_full)
xgb_preds_test = final_xgb.predict(X_test_full)
final_blended_log_preds = best_weight * lgbm_preds_test + (1 - best_weight) * xgb_preds_test
final_predictions = np.expm1(final_blended_log_preds)
final_predictions[final_predictions < 0] = 0.01
submission_df = pd.DataFrame({'sample_id': test_ids['sample_id'], 'price': final_predictions})
submission_df.to_csv('submission_GRANDMASTER.csv', index=False)

print("\n\n" + "="*50); print(" G R A N D   S U C C E S S !"); print("="*50)




[5a] Loading all of your SOTA feature files from Drive...

[5b] Aligning all training features...
Alignment complete. Training features shape: (74999, 2306)

[5c] Aligning all test features...
Final test matrix shape: (75000, 2306)

[5d] Training ensemble and calculating your SOTA SMAPE score...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 1.601177 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 587731
[LightGBM] [Info] Number of data points in the train set: 63749, number of used features: 2306
[LightGBM] [Info] Start training from score 2.711710


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:183: UserWarning: [16:32:06] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/core.py:2676: UserWarning: [16:32:39] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  if len(data.shape) != 1 and self.num_features() != data.shape[1]:
/usr/local/lib/python3.12/dist-packages/xgboost/core.py:729: UserWarning: [16:32:39] WARNING


##################################################
##  FINAL SOTA ENSEMBLE SMAPE: 43.3252%  ##
##################################################

[5e] Training final models on 100% of data for submission...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 2.124920 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 587767
[LightGBM] [Info] Number of data points in the train set: 74999, number of used features: 2306
[LightGBM] [Info] Start training from score 2.708050


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:183: UserWarning: [16:50:02] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  bst.update(dtrain, iteration=i, fobj=obj)



[5f] Blending predictions and creating your winning submission file...


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/xgboost/core.py:2676: UserWarning: [16:50:43] WARNING: /workspace/src/common/error_msg.cc:27: The tree method `gpu_hist` is deprecated since 2.0.0. To use GPU training, set the `device` parameter to CUDA instead.

    E.g. tree_method = "hist", device = "cuda"

  if len(data.shape) != 1 and self.num_features() != data.shape[1]:




 G R A N D   S U C C E S S !
